# **Part 1: Data Loading and Exploration**

In [83]:
from sklearn.datasets import fetch_california_housing

# Load the housing dataset
housing = fetch_california_housing()

In [84]:
# Import necessary libraries
import pandas as pd

# Create a Pandas DataFrame for the features and a Series for the target variable
X = pd.DataFrame(housing.data, columns=housing.feature_names) 
y = pd.Series(housing.target, name='med_house_value')


In [85]:
# Perform an initial exploration of the dataset

# Display the first five rows
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [86]:
# Print the feature names and check for missing values 
print(X.columns)
print(X.isnull().sum())

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64


In [87]:
# Generate summary statistics 
X.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


# **Part 2: Linear Regression on Unscaled Data** 

In [88]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Split the raw data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the linear regression model on unscaled data
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = lin_reg.predict(X_test)

In [89]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Unscaled Data Model:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Squared Error: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

Unscaled Data Model:
Mean Squared Error: 0.56
Root Squared Error: 0.75
R² Score: 0.58


In [90]:
# Print the coefficients and their associated features 
pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lin_reg.coef_
})

,Feature,Coefficient
0,MedInc,0.448675
1,HouseAge,0.009724
2,AveRooms,-0.123323
3,AveBedrms,0.783145
4,Population,-0.000002
5,AveOccup,-0.003526
6,Latitude,-0.419792
7,Longitude,-0.433708


## Interpretation Questions 
**What does the R² score tell us about model performance?**

The R² score tells us the proportion of the variance in the target variable explained by the model. If the score is closer to 1, this means there is a very good fit. If the score is closer to 0, the model fails to capture much variance. In this case, our target variable is 'med_house_value', and the R² score is 0.58. This means that the model explains 58% of the variation in housing prices. While this number is closer to 1 than 0, it is pretty moderate. 42% of the variation is still unexplained.

**Which features seem to have the strongest impact on predictions based on the model’s coefficients?**

In order to determine impact, you look at the absolute value of the coefficients. This data is unscaled, so there are not specific measurements or boundaries for impact. Stronger impact is based on the relative magnitude of the coefficients. AveBedrms, MedInc, Longitude, and Latitude, therefore, are the largest in terms of absolute value and have the strongest impact on predictions. 

**How well do the predicted values match the actual values?**

Based on both R² and RMSE, the predicted values are not extremely precise. The R² captures the general trend of the actual values, as 58% is closer to 1 than 0. The RMSE is 0.75, which means that predictions, on average, are about $75,000 off from the actual values. When thinking about house values in the hundreds of thousands range (~ $200,000-$500,000), this error substantial; while not grand,  it is certainly worthy of noticing. Furthermore, the predicted values are a moderate match as they capture the overall trend, but are not highly precise. 

# **Part 3: Feature Selection and Simplified Model**

**Select three features from the dataset to build a simplified model**

I am selecting "MedInc", "AveBedrms", and "Latitude". I want to select features that will still allow me to predict house value. I chose median income because areas with higher income will most likely have higher property values. The amount of bedrooms, on average, will give me a gauge of how many people can comfortably fit within the home. A larger number of bedrooms typically means a more expensive property. Lastly, the latitude will give insight into geographical region, and how properties are priced based on where they are. By using these three features I am considering economic factors, characteristics of the home itself, as well as geographical factors. 


In [91]:
# Choosing three features
three_features = ['MedInc', 'AveBedrms','Latitude']

# Selecting these three features for the model
X_train_features = X_train[three_features]
X_test_features = X_test[three_features]

# Train the new linear regression model
lin_reg_features = LinearRegression()
lin_reg_features.fit(X_train_features, y_train)

# Make predictions
y_pred_features = lin_reg_features.predict(X_test_features)

# Evaluate the new model
mse_features = mean_squared_error(y_test, y_pred_features)
rmse_features = root_mean_squared_error(y_test, y_pred_features)
r2_features = r2_score(y_test, y_pred_features)

print("Simplified Model (3 Features):")
print(f"Mean Squared Error: {mse_features:.2f}")
print(f"Root Mean Squared Error: {rmse_features:.2f}")
print(f"R² Score: {r2_features:.2f}")

Simplified Model (3 Features):
Mean Squared Error: 0.70
Root Mean Squared Error: 0.84
R² Score: 0.47


## Interpretation Questions 
**How does the simplified model compare to the full model?**

The simplified model, in terms of comparing actual to predicted values, is worse than the full model. The R² score decreased from 0.58 to 0.47, which means that the simplified model explains only 47% of the variation, which is a drop of 11% in explained variance. Also, the RMSE increased from 0.75 to 0.84, meaning the predicted values are $84,000 off from the actual values. This is an increase of $9,000 in error. However, it makes sense that the simplified model would perform inaccurately, as it uses fewer features and has less information than the full model. 

**Would you use this simplified model in practice? Why or why not?**

While the simplified model might be useful in terms of speed, efficiency, or interpreting certain features, I would still use the full model in practice. Assuming accuracy is priority, when comparing the two models, the predictive accuracy of the full model is much better. The explained variance is higher, at 58% compared to 47%, and prediction error lower, at 0.75 compared to 0.84.  